# Check the quality of a match

[All How-to Guides](README.md)

In observational data, control and treated units can differ before treatment. Check baseline imbalance on an important feature, then compare its values within the matched pairs.

## 1. Check baseline imbalance

In the simulated data, assignment depends on `X_lag_1`. Assign `TargetRole` to that feature to compare the original groups.

The generator defaults to `dependent_division=True`. The example spells it out to show where the baseline imbalance comes from.

In [1]:
from hypex import HomogeneityTest, Matching
from hypex.dataset import Dataset, FeatureRole, TargetRole, TreatmentRole
from hypex.utils.tutorial_data_creation import gen_control_variates_df

frame = gen_control_variates_df(
    data_size=1200, dependent_division=True, random_state=42
)
baseline = Dataset(
    roles={"Treatment": TreatmentRole(), "X_lag_1": TargetRole()},
    data=frame,
)
HomogeneityTest().execute(baseline).summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,X_lag_1,1,1.271988,3.459913,2.187925,172.008319,1.929678,2.446173,NOT OK,7.012739e-56,NOT OK,2.235408e-44


## 2. Match observations

Assign `FeatureRole` to the historical feature and `TargetRole` to the post-treatment outcome. Use a KS test to compare feature distributions and enable `extract_full_data` and `compute_indexes` to inspect matched values and indices.

The example explicitly uses the default `distance="mahalanobis"`, which accounts for scale and correlation when several features are present. Here one feature makes it easier to compare individual pairs.

`quality_tests="ks-test"` selects the diagnostic. This version requires both extraction flags to inspect the neighbors' indices and values. For `quality_results` or `summary` alone, leave both flags at their defaults.

In [2]:
data = Dataset(
    roles={
        "Treatment": TreatmentRole(),
        "Target": TargetRole(),
        "X_lag_1": FeatureRole(),
    },
    data=frame,
)
result = Matching(
    distance="mahalanobis",
    quality_tests="ks-test",
    compute_indexes=True,
    extract_full_data=True,
).execute(data)
result.quality_results

,feature,group,KSTest pass,KSTest p-value
0,X_lag_1,0,OK,0.999996
1,X_lag_1,1,OK,0.862405


## 3. Inspect individual pairs and effect estimates

In [3]:
result.indexes.iloc[:5]

,indexes_0
0,415
1,47
2,998
3,622
4,557


In [4]:
result.full_data.select(["X_lag_1", "X_lag_1_matched_0"]).iloc[:5]

,X_lag_1,X_lag_1_matched_0
0,1.899464,1.895593
1,0.472709,0.463486
2,9.207838,7.569159
3,7.090324,7.146040
4,-0.352612,-0.344385


In [5]:
result.summary

,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,1.9775,32.7515,0.9519,-62.2154,66.1704
ATC,-0.0869,30.8732,0.9978,-60.5984,60.4246
ATE,0.9608,26.3372,0.9709,-50.6601,52.5817


<a id="matching-mixed-checks"></a>

## 4. Choose checks for different feature types

On a small separate sample, derive a categorical feature from `X_lag_1` and pass several tests to `quality_tests`. With one neighbor, each check uses the pairs shown in the example.

Matching defaults to one neighbor and enables categorical encoding. `quality_tests` only controls which diagnostics run.

The example spells out the default `distance="mahalanobis"` so the distance is explicit across runs.

In [6]:
mixed_frame = frame.head(80).assign(
    segment=lambda rows: (rows["X_lag_1"] >= 2.5).map(
        {False: "low", True: "high"}
    )
)
mixed_data = Dataset(
    roles={
        "Treatment": TreatmentRole(),
        "Target": TargetRole(),
        "X_lag_1": FeatureRole(),
        "segment": FeatureRole(str),
    },
    data=mixed_frame,
)
mixed_match = Matching(
    distance="mahalanobis",
    quality_tests=["t-test", "ks-test", "chi2-test"],
).execute(mixed_data)
mixed_match.quality_results

,feature,group,TTest pass,TTest p-value,KSTest pass,KSTest p-value,Chi2Test pass,Chi2Test p-value
0,X_lag_1,0,OK,0.597214,OK,0.331948,NaN,NaN
1,X_lag_1,1,OK,0.574485,OK,0.690184,NaN,NaN
2,segment,0,NaN,NaN,NaN,NaN,OK,1.0
3,segment,1,NaN,NaN,NaN,NaN,OK,1.0


HypEx reports t-tests and KS tests for the numeric feature and a chi-square test for the categorical feature. Empty entries indicate that a test does not apply to that feature. This table describes the balance of the pairs found with these search settings.

## Interpret the results

Before matching, the difference in means for `X_lag_1` is about 2.19, and the KS test detects a difference in distributions. After matching, the two directional checks in `quality_results` both pass at 0.05. Use `indexes` to identify matched rows and `full_data` to compare feature values within each pair.

If differences still matter for your analysis, inspect overlap between the groups before interpreting ATE, ATT, and ATC in `summary`. A causal interpretation also requires a reason to believe that the analysis accounts for factors affecting both assignment and outcome.

The [in-depth Matching tutorial](../quick_tutorials/Matching.ipynb) covers the arguments and reports.